# 03 - Modeling

This notebook trains and evaluates machine learning models for purchase prediction.

In [ ]:
# =============================================================================
# Import Required Libraries
# =============================================================================
# Data manipulation
import pandas as pd
import numpy as np

# Scikit-learn: model building, preprocessing, and evaluation
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report

# LightGBM: gradient boosting for tabular data
from lightgbm import LGBMClassifier

In [ ]:
# =============================================================================
# Load Processed Intent Dataset
# =============================================================================
# Load prefix-based features from feature engineering notebook
# This dataset uses only early session behavior to avoid leakage

intent_df = pd.read_csv("../data/processed/intent_prefix_dataset.csv")
print(f"Loaded {intent_df.shape[0]} sessions with {intent_df.shape[1]} columns")
intent_df.head(), intent_df.shape

In [ ]:
# =============================================================================
# Define Target and Features
# =============================================================================
# Target: Binary label indicating quick purchase decision vs extended browsing

y = intent_df["target"]
X = intent_df.drop(columns=["target"], errors="ignore")

# Remove session identifier (not a feature)
X = X.drop(columns=["session_id"], errors="ignore")

# 🚨 CRITICAL: Remove leakage features
# These columns encode session length information which directly leaks the target
leakage_cols = [
    "session_len",      # Total session length (used to define target)
    "n_events_pfx",     # Event count (correlates with session length)
    "n_view_pfx",       # View count (same issue)
]

X = X.drop(columns=leakage_cols, errors="ignore")

print("Final feature columns:")
print(X.columns.tolist())
print(f"\nFeature shape: {X.shape}")

## Load Processed Data

In [ ]:
# =============================================================================
# Identify Feature Types
# =============================================================================
# Categorical features: mode of category preferences (need encoding)
# Numeric features: price stats and variety counts (can use directly)

cat_features = [c for c in X.columns if c.endswith("_mode_pfx")]
num_features = [c for c in X.columns if c not in cat_features]

print(f"Numeric features ({len(num_features)}): {num_features}")
print(f"Categorical features ({len(cat_features)}): {cat_features}")

In [ ]:
# =============================================================================
# Train/Test Split
# =============================================================================
# 80/20 split with stratification to maintain class balance

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples ({y_train.mean():.1%} positive)")
print(f"Test set: {X_test.shape[0]} samples ({y_test.mean():.1%} positive)")

In [ ]:
# =============================================================================
# Logistic Regression with Full Preprocessing Pipeline
# =============================================================================
# Pipeline includes:
# - StandardScaler for numeric features (improves convergence)
# - OneHotEncoder for categorical features
# - Logistic Regression with class balancing

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Define feature groups
num_features = [
    'price_mean_pfx', 'price_std_pfx', 'price_min_pfx', 'price_max_pfx',
    'price2_mean_pfx', 'price2_std_pfx',
    'country_nunique_pfx',
    'page_1_(main_category)_nunique_pfx',
    'page_2_(clothing_model)_nunique_pfx',
    'colour_nunique_pfx',
    'model_photography_nunique_pfx'
]

cat_features = [
    'country_mode_pfx',
    'page_1_(main_category)_mode_pfx',
    'page_2_(clothing_model)_mode_pfx',
    'colour_mode_pfx',
    'model_photography_mode_pfx'
]

# Build preprocessing transformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),                                    # Standardize numeric
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), cat_features),  # Encode categorical
    ]
)

# Build Logistic Regression pipeline with class balancing
log_reg = Pipeline([
    ("preprocess", preprocessor),
    ("clf", LogisticRegression(
        solver="saga",           # Supports L1/L2 and large datasets
        max_iter=5000,           # Ensure convergence
        class_weight="balanced", # Handle class imbalance
        n_jobs=-1
    ))
])

# Train and evaluate
log_reg.fit(X_train, y_train)
p_lr = log_reg.predict_proba(X_test)[:, 1]

print("=== Logistic Regression (with scaling) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, p_lr):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, p_lr):.4f}")

## Train Models

In [ ]:
# =============================================================================
# Preprocessing for Tree-Based Models
# =============================================================================
# Tree models don't need scaling; use passthrough for numeric features
# Still need one-hot encoding for categorical features

preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),                                      # No scaling needed
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_features),
    ]
)

In [ ]:
# =============================================================================
# Baseline: Simple Logistic Regression (without scaling)
# =============================================================================
# Compare against the scaled version to see impact of preprocessing

baseline = Pipeline([
    ("prep", preprocess),
    ("clf", LogisticRegression(max_iter=1000))
])

baseline.fit(X_train, y_train)
p_base = baseline.predict_proba(X_test)[:, 1]

print("=== Baseline: Logistic Regression (no scaling) ===")
print(f"ROC-AUC: {roc_auc_score(y_test, p_base):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, p_base):.4f}")

In [ ]:
# =============================================================================
# LightGBM: Gradient Boosting Model
# =============================================================================
# LightGBM with regularization to prevent overfitting
# Key hyperparameters tuned for this dataset size

lgbm = Pipeline([
    ("prep", preprocess),
    ("clf", LGBMClassifier(
        n_estimators=300,          # Number of boosting rounds
        learning_rate=0.05,        # Conservative step size
        num_leaves=31,             # Tree complexity
        max_depth=6,               # Limit depth to prevent overfitting
        min_child_samples=50,      # Minimum samples per leaf
        subsample=0.9,             # Row subsampling
        colsample_bytree=0.9,      # Feature subsampling
        reg_alpha=1.0,             # L1 regularization
        reg_lambda=1.0,            # L2 regularization
        random_state=42
    ))
])

lgbm.fit(X_train, y_train)
p_lgbm = lgbm.predict_proba(X_test)[:, 1]

print("=== LightGBM ===")
print(f"ROC-AUC: {roc_auc_score(y_test, p_lgbm):.4f}")
print(f"PR-AUC:  {average_precision_score(y_test, p_lgbm):.4f}")

## Evaluate Models

In [ ]:
# =============================================================================
# Classification Report for LightGBM
# =============================================================================
# Detailed precision/recall/F1 at default 0.5 threshold

y_pred = (p_lgbm >= 0.5).astype(int)
print("LightGBM Classification Report (threshold=0.5):")
print(classification_report(y_test, y_pred))

In [ ]:
# =============================================================================
# Save Model Performance Metrics
# =============================================================================
# Export metrics and predictions for thesis documentation

import os

results_dir = "../reports/results_tables"
os.makedirs(results_dir, exist_ok=True)

# Model comparison table
metrics_df = pd.DataFrame({
    "Model": ["Logistic Regression", "LightGBM"],
    "ROC_AUC": [roc_auc_score(y_test, p_base), roc_auc_score(y_test, p_lgbm)],
    "PR_AUC": [average_precision_score(y_test, p_base), average_precision_score(y_test, p_lgbm)],
})
metrics_df.to_csv(os.path.join(results_dir, "model_performance_metrics_optionB.csv"), index=False)
print("Saved metrics:")
print(metrics_df)

# Sample predictions for error analysis
preds_df = pd.DataFrame({
    "y_true": y_test.values,
    "logreg_prob": p_base,
    "lgbm_prob": p_lgbm
})
preds_df.to_csv(os.path.join(results_dir, "sample_predictions_optionB.csv"), index=False)
print(f"\nSaved predictions to: {results_dir}")

In [ ]:
# =============================================================================
# Save Final Modeling Dataset for Explainability
# =============================================================================
# Export the exact features used for training (no leakage columns)
# This ensures explainability notebook uses identical data

final_model_df = X.copy()
final_model_df["target"] = y.values

output_path = "../data/processed/modeling_dataset_optionB_FINAL.csv"
final_model_df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Columns: {final_model_df.columns.tolist()}")